# Subscription churn — exploratory pass

Working notebook. Not cleaned up. Several modelling attempts live here side by side; the one that shipped is the gradient-boosting block near the bottom.

In [1]:
import os, json, math, random, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)
random.seed(7); np.random.seed(7)

In [3]:
RAW = os.environ.get('CHURN_RAW', './data/raw')
INTERIM = './data/interim'
os.makedirs(INTERIM, exist_ok=True)

## Load

In [4]:
accounts = pd.read_parquet(f'{RAW}/accounts.parquet')
accounts.shape

In [5]:
events = pd.read_parquet(f'{RAW}/events.parquet')
events.shape

In [6]:
billing = pd.read_csv(f'{RAW}/billing.csv', parse_dates=['period_start', 'period_end'])
billing.shape

In [7]:
support = pd.read_json(f'{RAW}/support_tickets.json', lines=True)
support.shape

In [8]:
accounts.head()

In [9]:
events.dtypes

In [10]:
billing.describe(include='all').T

## First look at nulls

In [11]:
accounts.isna().mean().sort_values(ascending=False).head(20)

In [12]:
events.isna().mean().sort_values(ascending=False).head(20)

In [13]:
billing.isna().mean().sort_values(ascending=False)

In [14]:
# support tickets are sparse by design — most accounts never open one
support.account_id.nunique(), accounts.account_id.nunique()

In [15]:
accounts[accounts.plan.isna()].head(20)

In [16]:
accounts.plan.value_counts(dropna=False)

## Cleaning

Drop the columns that are more than 60% missing, then fix the obvious type problems.

In [17]:
thresh = 0.6
sparse_cols = accounts.columns[accounts.isna().mean() > thresh].tolist()
sparse_cols

In [18]:
accounts = accounts.drop(columns=sparse_cols)
accounts.shape

In [19]:
accounts['signup_date'] = pd.to_datetime(accounts['signup_date'], errors='coerce')
accounts['signup_date'].isna().sum()

In [20]:
accounts = accounts[accounts.signup_date.notna()].copy()
len(accounts)

In [21]:
accounts['plan'] = accounts['plan'].fillna('unknown').str.lower().str.strip()
accounts.plan.value_counts()

In [22]:
accounts['seats'] = pd.to_numeric(accounts['seats'], errors='coerce').fillna(1).astype(int)
accounts.seats.describe()

In [23]:
accounts['mrr'] = accounts['mrr'].clip(lower=0)
accounts.mrr.describe()

In [24]:
dupes = accounts.duplicated('account_id').sum()
dupes

In [25]:
accounts = accounts.drop_duplicates('account_id', keep='last')
len(accounts)

## Defining the label

Churn = no billing period starting in the 60 days after the observation cut.

In [17]:
CUT = pd.Timestamp('2026-03-01')
HORIZON = pd.Timedelta(days=60)

In [18]:
def churned(group, cut=CUT, horizon=HORIZON):
    future = group[group.period_start > cut]
    return int(future.empty or future.period_start.min() > cut + horizon)

In [28]:
label = billing.groupby('account_id').apply(churned).rename('churned')
label.mean()

In [29]:
accounts = accounts.merge(label, on='account_id', how='left')
accounts.churned = accounts.churned.fillna(1).astype(int)
accounts.churned.mean()

In [30]:
accounts.groupby('plan').churned.agg(['mean', 'size']).sort_values('mean', ascending=False)

## Event aggregates

In [31]:
events['ts'] = pd.to_datetime(events['ts'])
window = events[events.ts.between(CUT - pd.Timedelta(days=90), CUT)]
len(window)

In [32]:
by_account = window.groupby('account_id')
event_counts = by_account.size().rename('events_90d')
event_counts.describe()

In [33]:
active_days = by_account.ts.apply(lambda s: s.dt.date.nunique()).rename('active_days_90d')
active_days.describe()

In [34]:
last_seen = by_account.ts.max().rename('last_seen')
recency = (CUT - last_seen).dt.days.rename('recency_days')
recency.describe()

In [35]:
kinds = window.pivot_table(index='account_id', columns='kind', values='ts', aggfunc='count').fillna(0)
kinds.columns = [f'n_{c}' for c in kinds.columns]
kinds.shape

In [36]:
feat = accounts.set_index('account_id').join([event_counts, active_days, recency, kinds])
feat.shape

In [37]:
feat[['events_90d', 'active_days_90d', 'recency_days']] = feat[['events_90d', 'active_days_90d', 'recency_days']].fillna(0)
feat.isna().mean().sort_values(ascending=False).head()

## Support signal

In [38]:
support['opened_at'] = pd.to_datetime(support['opened_at'])
tickets = support[support.opened_at.between(CUT - pd.Timedelta(days=180), CUT)]
len(tickets)

In [39]:
ticket_counts = tickets.groupby('account_id').size().rename('tickets_180d')
escalations = tickets[tickets.severity.isin(['high', 'critical'])].groupby('account_id').size().rename('escalations_180d')

In [ ]:
feat = feat.join([ticket_counts, escalations])
feat[['tickets_180d', 'escalations_180d']] = feat[['tickets_180d', 'escalations_180d']].fillna(0)

In [ ]:
feat.groupby(pd.cut(feat.tickets_180d, [-1, 0, 1, 3, 100])).churned.mean()

## Billing trend

In [40]:
recent_billing = billing[billing.period_start <= CUT].sort_values('period_start')
last_three = recent_billing.groupby('account_id').tail(3)

In [41]:
trend = last_three.groupby('account_id').amount.agg(['mean', 'std', 'last'])
trend.columns = ['bill_mean', 'bill_std', 'bill_last']
trend.head()

In [42]:
trend['bill_delta'] = trend.bill_last - trend.bill_mean
feat = feat.join(trend)
feat.shape

In [43]:
feat[[c for c in feat.columns if c.startswith('bill_')]] = feat[[c for c in feat.columns if c.startswith('bill_')]].fillna(0)

## Plots

In [44]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 4)

In [45]:
feat.recency_days.hist(bins=40)
plt.title('Recency at cut')
plt.show()

In [46]:
feat.groupby('plan').churned.mean().sort_values().plot.barh()
plt.title('Churn rate by plan')
plt.show()

In [47]:
fig, ax = plt.subplots()
for label_value, group in feat.groupby('churned'):
    ax.hist(group.active_days_90d, bins=30, alpha=0.5, label=f'churned={label_value}')
ax.legend()
plt.show()

In [48]:
corr = feat.select_dtypes('number').corr()['churned'].sort_values()
corr.head(12)

In [49]:
corr.tail(12)

## Train / test split

In [50]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

In [42]:
numeric = feat.select_dtypes('number').drop(columns=['churned'])
X = pd.get_dummies(feat[['plan']]).join(numeric)
y = feat.churned
X.shape, y.mean()

In [52]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=7)
X_train.shape, X_test.shape

In [53]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

## Attempt 1 — logistic regression

In [54]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [55]:
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))
logit.fit(X_train, y_train)

In [56]:
p = logit.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, p), average_precision_score(y_test, p)

In [57]:
cross_val_score(logit, X_train, y_train, cv=cv, scoring='roc_auc').mean()

In [58]:
coefs = pd.Series(logit[-1].coef_[0], index=X.columns).sort_values()
coefs.head(10)

In [59]:
coefs.tail(10)

## Attempt 2 — random forest

Left in for comparison; it overfits badly at default depth.

In [60]:
from sklearn.ensemble import RandomForestClassifier

In [61]:
rf = RandomForestClassifier(n_estimators=400, random_state=7, n_jobs=-1)
rf.fit(X_train, y_train)

In [62]:
roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

In [63]:
roc_auc_score(y_train, rf.predict_proba(X_train)[:, 1])  # note the gap

In [64]:
rf_shallow = RandomForestClassifier(n_estimators=400, max_depth=8, min_samples_leaf=20, random_state=7, n_jobs=-1)
rf_shallow.fit(X_train, y_train)
roc_auc_score(y_test, rf_shallow.predict_proba(X_test)[:, 1])

In [65]:
pd.Series(rf_shallow.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)

## Attempt 3 — gradient boosting (this is the one)

In [66]:
from sklearn.ensemble import HistGradientBoostingClassifier

In [67]:
LEARNING_RATE = 0.06
MAX_LEAF_NODES = 31
MIN_SAMPLES_LEAF = 25
L2 = 1.0
MAX_ITER = 400

In [68]:
gbm = HistGradientBoostingClassifier(
    learning_rate=LEARNING_RATE,
    max_leaf_nodes=MAX_LEAF_NODES,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    l2_regularization=L2,
    max_iter=MAX_ITER,
    early_stopping=True,
    random_state=7,
)

In [69]:
gbm.fit(X_train, y_train)
gbm.n_iter_

In [70]:
p_gbm = gbm.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, p_gbm), average_precision_score(y_test, p_gbm)

In [71]:
print(classification_report(y_test, (p_gbm > 0.5).astype(int), digits=3))

### Threshold sweep

In [72]:
import numpy as np
rows = []
for t in np.arange(0.1, 0.9, 0.05):
    pred = (p_gbm > t).astype(int)
    tp = ((pred == 1) & (y_test == 1)).sum()
    fp = ((pred == 1) & (y_test == 0)).sum()
    fn = ((pred == 0) & (y_test == 1)).sum()
    rows.append((t, tp / max(tp + fp, 1), tp / max(tp + fn, 1)))
sweep = pd.DataFrame(rows, columns=['threshold', 'precision', 'recall'])

In [ ]:
sweep.plot(x='threshold')
plt.title('Precision / recall against threshold')
plt.show()

In [ ]:
sweep.assign(f1=lambda d: 2 * d.precision * d.recall / (d.precision + d.recall)).sort_values('f1', ascending=False).head()

In [ ]:
THRESHOLD = 0.42

## Calibration

In [73]:
from sklearn.calibration import calibration_curve
prob_true, prob_pred = calibration_curve(y_test, p_gbm, n_bins=12)

In [74]:
plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0, 1], [0, 1], '--')
plt.title('Calibration')
plt.show()

In [75]:
from sklearn.calibration import CalibratedClassifierCV
calibrated = CalibratedClassifierCV(gbm, method='isotonic', cv='prefit')
calibrated.fit(X_test, y_test)

In [76]:
p_cal = calibrated.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, p_cal)

## Segment checks

Does the model behave the same on the segments the CS team actually works?

In [77]:
segments = feat.loc[X_test.index]
segments['p'] = p_gbm

In [78]:
segments.groupby('plan').apply(lambda g: roc_auc_score(g.churned, g.p) if g.churned.nunique() > 1 else np.nan)

In [79]:
segments.groupby(pd.qcut(segments.mrr, 4)).apply(lambda g: roc_auc_score(g.churned, g.p) if g.churned.nunique() > 1 else np.nan)

In [80]:
segments.groupby(pd.cut(segments.seats, [0, 1, 5, 25, 10_000])).churned.mean()

In [81]:
segments[segments.plan == 'unknown'].shape

## Error analysis

In [82]:
segments['pred'] = (segments.p > THRESHOLD).astype(int)
false_neg = segments[(segments.pred == 0) & (segments.churned == 1)]
len(false_neg)

In [83]:
false_neg[['plan', 'mrr', 'seats', 'recency_days', 'active_days_90d', 'tickets_180d']].describe()

In [84]:
false_pos = segments[(segments.pred == 1) & (segments.churned == 0)]
false_pos[['plan', 'mrr', 'recency_days']].head(20)

In [85]:
false_neg.plan.value_counts(normalize=True)

## Export

In [86]:
import joblib
joblib.dump(gbm, f'{INTERIM}/churn_gbm.joblib')

In [ ]:
scored = pd.DataFrame({'account_id': X_test.index, 'p_churn': p_gbm})
scored.to_csv(f'{INTERIM}/scored_holdout.csv', index=False)

In [ ]:
with open(f'{INTERIM}/model_card.json', 'w') as fh:
    json.dump({'threshold': THRESHOLD, 'auc': float(roc_auc_score(y_test, p_gbm)), 'n_train': int(len(X_train))}, fh, indent=2)

In [ ]:
os.listdir(INTERIM)

## Scratch

Everything below is leftovers. Kept because re-deriving it is slower than scrolling past it.

In [ ]:
# old feature idea — weekly seat churn. Never finished.
# weekly = events.set_index('ts').groupby('account_id').resample('W').size()

In [ ]:
feat.columns.tolist()[:40]

In [ ]:
feat.columns.tolist()[40:]

In [ ]:
len(feat.columns)

In [ ]:
feat.churned.value_counts()

In [ ]:
X_train.dtypes.value_counts()

In [ ]:
# sanity: no leakage from billing after the cut
assert billing[billing.period_start > CUT].account_id.isin(X_train.index).any()

In [ ]:
del rf
import gc; gc.collect()

In [ ]:
print('done')